# Bài 1.2 - Tìm "vách đá" của quantization (CIFAR-10)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KuSenpai/modelopt101/blob/exercise-1.2-quant-cliff/exercises/1.2_quant_cliff.ipynb)

**Mục tiêu.** Lấy student CNN của bài 1.1, giảm dần số bit của weight từ 8 xuống 2 và tìm
mức bit mà accuracy bắt đầu "rơi xuống vực". So sánh hai cách chọn scale:

- **per-tensor:** cả tensor weight dùng chung một scale;
- **per-channel:** mỗi output channel (mỗi filter) có scale riêng.

Phần làm thêm: viết **QAT (quantization-aware training)** bằng **straight-through estimator (STE)**
rồi so với PTQ ở 4 bit.

```
student Net (bài 1.1, train có KD từ ResNet-18)
   ├─ (b) PTQ weight fake-quant: bits = 8 → 2, per-tensor vs per-channel
   └─ (c) QAT per-channel bằng STE: INT4 (+ INT3, INT2 làm bonus)
```

**Chỉ tiêu:** QAT INT4 mất **≤ 1 điểm %** accuracy so với FP32.

**Runtime:** `Runtime -> Change runtime type -> GPU (T4)`, rồi `Runtime -> Run all`.
Khoảng 35 phút, chưa tính thời gian tải CIFAR-10.

## 1. Setup

In [ ]:
import os, io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.parametrize as P
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"     # mixed precision: ~2x faster training on T4
print("device:", torch.cuda.get_device_name(0) if USE_AMP else "cpu")

# --- knobs you can play with ---
EPOCHS_TEACHER  = 15
EPOCHS_STUDENT  = 20
QAT_EPOCHS      = 8
BATCH_SIZE      = 256
LR              = 0.1
QAT_LR          = 0.01          # low LR: QAT only nudges an already-trained model
T_KD, ALPHA_KD  = 4.0, 0.7      # distillation temperature / weight on the teacher term
BITS            = [8, 7, 6, 5, 4, 3, 2]
QAT_BITS        = [4, 3, 2]     # 4 is the target, 3 and 2 are the bonus

# checkpoints: Google Drive if it is mounted, otherwise local (/content on Colab)
CKPT_DIR = "/content/drive/MyDrive/modelopt101_ckpt" if os.path.isdir("/content/drive/MyDrive") else "./ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

## 2. Data - CIFAR-10

Giống hệt bài 1.1: random crop có padding + lật ngang, normalize theo mean/std của CIFAR-10.

In [ ]:
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(mean, std)])
test_tf  = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2,
                          pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_set,  batch_size=512,        shuffle=False, num_workers=2,
                          pin_memory=True)
print(len(train_set), "train /", len(test_set), "test images")

## 3. Models

Cùng hai kiến trúc với bài 1.1: teacher ResNet-18 (stem CIFAR) và student CNN kiểu VGG 6 conv.

In [ ]:
def make_teacher(num_classes=10):
    m = torchvision.models.resnet18(num_classes=num_classes)
    m.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)   # CIFAR stem
    m.maxpool = nn.Identity()
    return m

def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Net(nn.Module):
    def __init__(self, widths=(64, 64, 128, 128, 256, 256), num_classes=10):
        super().__init__()
        w = widths
        self.features = nn.Sequential(
            conv_block(3,    w[0]), conv_block(w[0], w[1]), nn.MaxPool2d(2),   # 32 -> 16
            conv_block(w[1], w[2]), conv_block(w[2], w[3]), nn.MaxPool2d(2),   # 16 -> 8
            conv_block(w[3], w[4]), conv_block(w[4], w[5]), nn.MaxPool2d(2),   # 8  -> 4
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(w[5], num_classes))
        self.widths = tuple(widths)

    def forward(self, x):
        return self.head(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"teacher params: {count_params(make_teacher()):,}")
print(f"student params: {count_params(Net()):,}")

## 4. Train / eval / quantization utilities

`train`, `evaluate`, `distillation_loss`, `quantize` / `dequantize` / `quantize_model` chép nguyên
từ bài 1.1. Thêm `train_or_load` để lưu checkpoint: chạy lại notebook thì không phải train lại.

In [ ]:
def train(model, loss_fn, epochs, lr=LR, tag=""):
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    for ep in range(epochs):
        t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=USE_AMP):
                loss = loss_fn(model(x), x, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        print(f"[{tag}] epoch {ep+1:02d}/{epochs}  test_acc={evaluate(model):.2f}%  ({time.time()-t0:.0f}s)")
    return model

@torch.no_grad()
def evaluate(model):
    model.to(device).eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    model.train()
    return 100.0 * correct / total

def model_size_mb(state_dict):
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getbuffer().nbytes / 1e6

def train_or_load(name, model, loss_fn, epochs, **kw):
    path = os.path.join(CKPT_DIR, name + ".pt")
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location="cpu"))
        print(f"[{name}] loaded checkpoint {path}")
    else:
        train(model, loss_fn, epochs, tag=name, **kw)
        torch.save(model.state_dict(), path)
        print(f"[{name}] saved checkpoint {path}")
    return model.to(device).eval()

ce_loss = lambda out, x, y: F.cross_entropy(out, y)

def distillation_loss(student_logits, teacher_logits, labels, T=T_KD, alpha=ALPHA_KD):
    ce_term = F.cross_entropy(student_logits, labels)
    kd_term = F.kl_div(F.log_softmax(student_logits.float() / T, dim=1),
                       F.softmax(teacher_logits.float() / T, dim=1),
                       reduction="batchmean") * T ** 2
    return alpha * kd_term + (1 - alpha) * ce_term

def quantize(x, n_bits=8, symmetric=True, per_channel=False):
    qmin, qmax = -(2 ** (n_bits - 1)), 2 ** (n_bits - 1) - 1
    if per_channel:
        flat = x.flatten(1)
        xmin, xmax = flat.min(1).values, flat.max(1).values
    else:
        xmin, xmax = x.min(), x.max()
    if symmetric:
        scale = (torch.maximum(xmin.abs(), xmax.abs()) / qmax).clamp(min=1e-8)
        zero_point = torch.zeros_like(scale)
    else:
        scale = ((xmax - xmin) / (qmax - qmin)).clamp(min=1e-8)
        zero_point = qmin - torch.round(xmin / scale)
    shape = [-1] + [1] * (x.dim() - 1)
    s = scale.view(shape) if per_channel else scale
    z = zero_point.view(shape) if per_channel else zero_point
    q = torch.clamp(torch.round(x / s) + z, qmin, qmax)
    return q, scale, zero_point

def dequantize(q, scale, zero_point, per_channel=False):
    if per_channel:
        shape = [-1] + [1] * (q.dim() - 1)
        scale, zero_point = scale.view(shape), zero_point.view(shape)
    return scale * (q - zero_point)

QUANTIZABLE = (nn.Conv2d, nn.Linear)

def quantize_model(model, n_bits=8, per_channel=True):
    """Return (packed_state_dict, dequantized_model) for weight-only PTQ."""
    packed = {}
    deq = copy.deepcopy(model).cpu().eval()
    qnames = [n for n, m in deq.named_modules() if isinstance(m, QUANTIZABLE)]
    for name, mod in deq.named_modules():
        if isinstance(mod, QUANTIZABLE):
            q, s, z = quantize(mod.weight.data, n_bits=n_bits, per_channel=per_channel)
            packed[name + ".weight_int"]   = q.to(torch.int8)
            packed[name + ".weight_scale"] = s.clone()
            mod.weight.data = dequantize(q, s, z, per_channel=per_channel)
    for k, v in deq.state_dict().items():
        if k not in {n + ".weight" for n in qnames}:
            packed[k] = v
    return packed, deq

## 5. (a) Model FP32 - student `Net` của bài 1.1, train **có KD**

Làm lại đúng công thức bài 1.1: train teacher ResNet-18 15 epoch, rồi train student `Net`
(widths 64-64-128-128-256-256) 20 epoch bằng KD loss (T = 4, α = 0.7). Đây là model "1b. student, KD"
của bài 1.1 (lần đó đạt 91.38%). Model này làm **mốc FP32** cho mọi phép so sánh bên dưới.

Ở đây dùng student chưa pruning, để đường cong theo bit chỉ phản ánh quantization chứ không cộng
dồn sai số của pruning. Checkpoint lưu ở `CKPT_DIR`.

In [ ]:
teacher = train_or_load("teacher_resnet18", make_teacher(), ce_loss, EPOCHS_TEACHER)
print(f"teacher acc: {evaluate(teacher):.2f}%")
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

def kd_loss_fn(student_out, x, y):
    with torch.no_grad():
        teacher_out = teacher(x)
    return distillation_loss(student_out, teacher_out, y)

student = train_or_load("student_kd", Net(), kd_loss_fn, EPOCHS_STUDENT)
fp32_acc = evaluate(student)
print(f"\nFP32 student (KD) acc: {fp32_acc:.2f}%   size: {model_size_mb(student.state_dict()):.2f} MB")

## 6. (b) PTQ: quét bits 8 → 2, per-tensor vs per-channel

Weight-only fake-quant, symmetric: mọi `Conv2d` và `Linear` (kể cả conv đầu và lớp cuối) được
làm tròn về lưới `n_bits` rồi dequantize về FP32. BatchNorm và activation giữ FP32.

Với symmetric `n_bits`, lưới có các mức nguyên trong `[-2^(b-1), 2^(b-1) - 1]`, nên 2 bit chỉ còn
4 mức `{-2, -1, 0, 1} × scale`. Per-tensor dùng một scale `max|W| / qmax` cho cả lớp; chỉ cần một
filter có weight lớn là mọi filter khác bị làm tròn thô.

"Vách đá" = mức bit đầu tiên (đi từ 8 xuống) mà accuracy giảm **quá 1 điểm** và **quá 5 điểm** so với FP32.

In [ ]:
ptq = {"per-tensor": {}, "per-channel": {}}
for b in BITS:
    for gran in ptq:
        ptq[gran][b] = evaluate(quantize_model(student, n_bits=b, per_channel=(gran == "per-channel"))[1])

def first_drop(accs, thr):
    for b in BITS:                          # BITS goes 8 -> 2
        if fp32_acc - accs[b] > thr:
            return b
    return None

print(f"FP32: {fp32_acc:.2f}%\n")
print(f"{'bits':>4} | {'per-tensor':>10} {'Δ':>7} | {'per-channel':>11} {'Δ':>7}")
print("-" * 48)
for b in BITS:
    t, c = ptq["per-tensor"][b], ptq["per-channel"][b]
    print(f"{b:>4} | {t:>9.2f}% {t-fp32_acc:>+7.2f} | {c:>10.2f}% {c-fp32_acc:>+7.2f}")

cliff = {g: {thr: first_drop(ptq[g], thr) for thr in (1, 5)} for g in ptq}
print()
for g in ptq:
    print(f"{g:<12} cliff: first bit with >1 pt drop = {cliff[g][1]}, >5 pt drop = {cliff[g][5]}")

colors = {"per-tensor": "#d9534f", "per-channel": "#5b8def"}
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.axhline(fp32_acc, color="gray", ls="--", lw=1, label=f"FP32 ({fp32_acc:.2f}%)")
ax.axhspan(fp32_acc - 1, fp32_acc, color="green", alpha=0.08, label="within 1 pt of FP32")
for g, accs in ptq.items():
    ax.plot(BITS, [accs[b] for b in BITS], "o-", color=colors[g], label=g)
    for thr, marker in ((1, "v"), (5, "X")):
        b = cliff[g][thr]
        if b is not None:
            ax.plot(b, accs[b], marker, ms=13, mfc="none", mec=colors[g], mew=2)
            ax.annotate(f">{thr} pt @ {b}b", (b, accs[b]), textcoords="offset points",
                        xytext=(8, -14 if g == "per-tensor" else 6), color=colors[g], fontsize=9)
ax.set_xticks(BITS); ax.invert_xaxis()
ax.set_xlabel("weight bits"); ax.set_ylabel("test accuracy (%)")
ax.set_title("PTQ weight-only: accuracy vs bits  (▽ first >1 pt drop, ✕ first >5 pt drop)")
ax.set_ylim(0, 100); ax.grid(alpha=0.3); ax.legend(loc="lower left")
plt.tight_layout(); plt.show()

## 7. (c) QAT bằng straight-through estimator

Khi train, forward dùng weight đã fake-quant $\hat w = \mathrm{deq}(\mathrm{q}(w))$, nhưng `round` có
đạo hàm bằng 0 gần như khắp nơi nên gradient không đi qua được. STE coi phép làm tròn là hàm đồng
nhất khi backward:

$$\hat w_{\text{STE}} = w + \big(\mathrm{deq}(\mathrm{q}(w)) - w\big).\mathrm{detach}()
\quad\Rightarrow\quad \text{forward} = \hat w,\;\; \frac{\partial \hat w_{\text{STE}}}{\partial w} = 1$$

Model giữ một bản weight FP32 "ẩn" $w$, optimizer cập nhật $w$, còn mạng luôn nhìn thấy $\hat w$.
Nhờ vậy mạng học cách chịu được sai số làm tròn. Cài đặt bằng `torch.nn.utils.parametrize`: mỗi
`Conv2d`/`Linear` được gắn module `FakeQuantSTE` (per-channel), `layer.weight` trả về $\hat w_{\text{STE}}$.

Fine-tune từ checkpoint FP32, `QAT_EPOCHS` epoch, LR thấp (`QAT_LR`), dùng cùng KD loss như lúc train student.
Sau QAT, bỏ parametrization để lấy lại $w$ rồi đưa qua đúng `quantize_model` của phần PTQ, tức là
accuracy QAT được đo trên **cùng một kiểu model INT-n** với PTQ.

In [ ]:
class FakeQuantSTE(nn.Module):
    """Parametrization: weight -> per-channel symmetric fake-quant, identity gradient (STE)."""
    def __init__(self, n_bits):
        super().__init__()
        self.n_bits = n_bits

    def forward(self, w):
        w_q = dequantize(*quantize(w, self.n_bits, per_channel=True), per_channel=True)
        return w + (w_q - w).detach()

def qat(fp32_model, n_bits, epochs=QAT_EPOCHS, lr=QAT_LR):
    m = copy.deepcopy(fp32_model)
    layers = [mod for mod in m.modules() if isinstance(mod, QUANTIZABLE)]
    for mod in layers:
        P.register_parametrization(mod, "weight", FakeQuantSTE(n_bits))
    print(f"[QAT-INT{n_bits}] start (fake-quant, no training): {evaluate(m):.2f}%")
    train(m, kd_loss_fn, epochs, lr=lr, tag=f"QAT-INT{n_bits}")
    fq_acc = evaluate(m)
    for mod in layers:      # drop the fake-quant wrapper, keep the latent FP32 weights
        P.remove_parametrizations(mod, "weight", leave_parametrized=False)
    packed, deq = quantize_model(m, n_bits=n_bits, per_channel=True)
    acc = evaluate(deq)
    print(f"[QAT-INT{n_bits}] fake-quant acc {fq_acc:.2f}%  ->  exported INT{n_bits} model acc {acc:.2f}%")
    return acc

qat_acc = {}
for b in QAT_BITS:
    qat_acc[b] = qat(student, b)
    print()

## 8. (d) Tổng kết

In [ ]:
print(f"{'model':<26}{'acc':>9}{'Δ vs FP32':>11}")
print("-" * 46)
print(f"{'FP32 (student KD)':<26}{fp32_acc:>8.2f}%{0:>+11.2f}")
for b in QAT_BITS:
    for name, acc in ((f"PTQ INT{b} per-tensor", ptq["per-tensor"][b]),
                      (f"PTQ INT{b} per-channel", ptq["per-channel"][b]),
                      (f"QAT INT{b} per-channel", qat_acc[b])):
        print(f"{name:<26}{acc:>8.2f}%{acc-fp32_acc:>+11.2f}")
    print()

drop4 = fp32_acc - qat_acc[4]
print(f"target: QAT INT4 drop <= 1 pt  ->  drop = {drop4:.2f} pt  ->  {'PASS' if drop4 <= 1 else 'FAIL'}")
print(f"QAT INT4 recovers {qat_acc[4] - ptq['per-channel'][4]:+.2f} pt over PTQ INT4 per-channel")

x = np.arange(len(QAT_BITS)); w = 0.27
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (name, vals, c) in enumerate((
        ("PTQ per-tensor",  [ptq["per-tensor"][b]  for b in QAT_BITS], "#d9534f"),
        ("PTQ per-channel", [ptq["per-channel"][b] for b in QAT_BITS], "#5b8def"),
        ("QAT per-channel", [qat_acc[b]            for b in QAT_BITS], "#5cb85c"))):
    bars = ax.bar(x + (i - 1) * w, vals, w, label=name, color=c)
    ax.bar_label(bars, fmt="%.1f", fontsize=8)
ax.axhline(fp32_acc, color="gray", ls="--", lw=1, label=f"FP32 ({fp32_acc:.2f}%)")
ax.set_xticks(x, [f"INT{b}" for b in QAT_BITS]); ax.set_ylim(0, 100)
ax.set_ylabel("test accuracy (%)"); ax.set_title("PTQ vs QAT (weight-only)")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

## 9. Nhận xét

*(điền sau khi chạy trên Colab)*